# 02-01_統計データ加工
## 目的
- グラフデータの元となるCSVを計算し、列を追加

## 作成CSVデータ
- 元のCSVのヘッダー情報は以下
  - タイムスタンプ
  - 回線番号
  - 下りのトラヒック量(Byte)
  - 上りのトラヒック量(Byte)
  - 下りの廃棄量(Packets)
  - 下りの廃棄量(Byte)
- 計算の結果、追加となるCSVのヘッダー情報は以下
  - 下りトラヒック(Mbps)
  - 下り廃棄量(Mbps)
  - 下り合計(Mbps)

In [ ]:
import pandas as pd
from pathlib import Path

# --- 設定 ---
CONFIG = {
    "INPUT_FILE": "input/sample-traffic.csv",
    "OUTPUT_DIR": Path("output"),
    "INTERVAL_SEC": 300,
}

def process_traffic_data():
    # 1. データの読み込み
    if not Path(CONFIG["INPUT_FILE"]).exists():
        print(f"❌ エラー: {CONFIG['INPUT_FILE']} が見つかりません。")
        return

    df = pd.read_csv(CONFIG["INPUT_FILE"])
    df['タイムスタンプ'] = pd.to_datetime(df['タイムスタンプ'])

    # 2. 単位変換（Mbps）と合計値の計算
    # 変換係数: (Byte * 8) / (10^6 * interval)
    bits_to_mbps = 8 / (10**6 * CONFIG["INTERVAL_SEC"])
    
    df['下りトラヒック(Mbps)'] = df['下りのトラヒック量(Byte)'] * bits_to_mbps
    df['下り廃棄量(Mbps)'] = df['下りの廃棄量(Byte)'] * bits_to_mbps
    df['下り合計(Mbps)'] = df['下りトラヒック(Mbps)'] + df['下り廃棄量(Mbps)']

    # 3. 出力ディレクトリの準備
    CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

    # 4. 回線番号ごとにグループ化して保存
    grouped = df.groupby('回線番号')

    print(f"{len(grouped)} 件のパイプデータを処理中...")

    for pipe_name, group in grouped:
        # ソート（元データが時間順なら省略可能ですが念のため）
        group = group.sort_values('タイムスタンプ')

        # ファイル名に使用できない記号をクレンジング
        safe_name = str(pipe_name).replace("/", "_").replace(" ", "_")
        output_path = CONFIG["OUTPUT_DIR"] / f"{safe_name}.csv"
        
        group.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"✅ 全てのCSVを '{CONFIG['OUTPUT_DIR']}' に保存しました。")

if __name__ == "__main__":
    process_traffic_data()